In [ ]:
from pathlib import Path
import shutil, os
# Reuse the pinned source and cached downloads on the attached Volume.
shutil.copytree("/mnt/m3diff/baseline_reproduction_v1/repo", "/root/M3Diff", ignore=shutil.ignore_patterns(".git", "__pycache__"))


In [ ]:
import os, sys, json, shutil, subprocess, glob, time
VOL   = "/mnt/m3diff"
REPO  = "/root/M3Diff"
VENV  = "/root/m3diff-venv"
PY    = f"{VENV}/bin/python"
CKPT  = "/root/ckpt/M3Diff"
SYN   = f"{VOL}/synth"            # Notebook 1
FT    = f"{VOL}/ft"               # Notebook này
CODE  = f"{FT}/code"
OUT   = f"{FT}/m3diff-ft-lora"    # tên thư mục phải chứa 'm3diff' và 'lora' để repo nạp đúng kiểu LoRA
EVAL  = f"{FT}/eval"
OBJ_ALL   = f"{FT}/objects_all"
TRAIN_SYN = f"{FT}/train_synth"
CAMTEST   = f"{FT}/camtest"
MIX       = f"{FT}/train_mix.json"
QUANT, DTYPE = "none", "bfloat16"

# Siêu tham số
SCALE    = 1.0        # nhân số cặp train tổng hợp (1.0 ≈ 2.700 cặp)
N_REPLAY = 2700       # số câu OmniDiff train gốc trộn vào
LORA_R, LORA_ALPHA = 64, 128
LR, PROJ_LR, EPOCHS = 4e-5, 1e-5, 1
HF_TOKEN = os.environ.get("HF_TOKEN", "")
os.environ["MPLBACKEND"] = "Agg"
os.environ["TQDM_MININTERVAL"] = "20"
for d in [CODE, EVAL, f"{FT}/results"]:
    os.makedirs(d, exist_ok=True)

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "KHÔNG CÓ GPU")
mem = int(gpu.split(",")[-1]) if gpu else 0
BATCH = 4 if mem >= 70000 else (2 if mem >= 39000 else 1)
ACCUM = 16 // BATCH
print(f"batch/GPU = {BATCH}, tích luỹ gradient = {ACCUM} (batch hiệu dụng 16)")
if mem < 39000:
    print("!! GPU dưới 40GB: train có thể hết bộ nhớ. Nên chọn A100.")

need = {"code Notebook 1": f"{SYN}/code/make_synth.py", "bộ test tổng hợp": f"{SYN}/data/manifest.jsonl",
        "câu model gốc trên bộ tổng hợp": f"{SYN}/data/answers.jsonl", "điểm model gốc (per_pair)": f"{SYN}/results/per_pair.csv",
        "vật thể test": f"{SYN}/objects/objects.csv", "kết quả gốc Real": f"{VOL}/results/OmniDiff_Real/OmniDiff_Real.jsonl"}
for k, v in need.items():
    print(("OK    " if os.path.exists(v) else "THIẾU ") + f"{k}: {v}")
print(("OK    " if os.path.exists(f"{VOL}/results/OmniDiff_Render/OmniDiff_Render.jsonl") else "(không có, bỏ qua Render) ")
      + "kết quả gốc Render")

In [ ]:
%pip install -q uv
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")

if not os.path.exists(REPO):
    !git clone -q https://github.com/Yuan-Liu-01/M3Diff.git {REPO}
if not os.path.exists(PY):
    !{UV} venv {VENV} --python 3.10 -q

%cd {REPO}
!{UV} pip install -q --python {PY} -e ".[eval]" "numpy<2" accelerate==0.29.1 bitsandbytes==0.43.3 install-jdk
!{PY} -c "import torch, transformers, numpy; print('torch', torch.__version__, '| transformers', transformers.__version__, '| numpy', numpy.__version__); print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0), '|', round(torch.cuda.get_device_properties(0).total_memory/2**30, 1), 'GB')"
!{UV} pip install -q --python {PY} peft==0.4.0 "numpy<2" "opencv-python-headless<4.12" pandas scipy matplotlib pycocotools
!{PY} -c "import peft, numpy, cv2; print('peft', peft.__version__, '| numpy', numpy.__version__, '| cv2', cv2.__version__)"

In [ ]:
!{UV} pip install -q --python {PY} setuptools
!{PY} -c "import peft, numpy, cv2; print('peft', peft.__version__, '| numpy', numpy.__version__, '| cv2', cv2.__version__)"

In [ ]:
import hashlib
os.makedirs('/root/ckpt', exist_ok=True)
os.symlink('/mnt/m3diff/baseline_cache/M3Diff', CKPT)
os.makedirs(f'{REPO}/datasets', exist_ok=True)
os.symlink('/mnt/m3diff/baseline_cache/OmniDiff', f'{REPO}/datasets/OmniDiff')
assert os.path.isfile(f'{CKPT}/generation_config.json')
assert os.path.isfile(f'{CAMTEST}/questions.json')
print('Bước 3: dùng checkpoint và ảnh đã cache trên Volume; không tải lại.')


In [ ]:
path = f"{REPO}/llava/eval/model_vqa_qwen_OmniDiff_Real.py"
src = open(path).read()

# --- Patch 1: nén model cho vừa T4 ---
old1 = """    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
    )"""
new1 = """    quant_mode = os.environ.get("M3DIFF_QUANT", "8bit")
    skip = ["vision_tower", "vision_resampler", "mm_projector", "lm_head"]
    extra = {}
    if quant_mode == "8bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_8bit=True, llm_int8_skip_modules=skip)
    elif quant_mode == "4bit":
        extra["quantization_config"] = transformers.BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            llm_int8_skip_modules=skip)
    print(f"[patch] quant_mode={quant_mode}", flush=True)
    tokenizer, model, image_processor, _ = load_pretrained_model(
        model_path,
        args.model_base,
        model_name,
        use_flash_attn=args.flash_attention,
        torch_dtype=torch_dtype,
        device_map={"": 0},
        **extra,
    )"""

# --- Patch 2: chạy tiếp từ chỗ dừng + ghi từng câu ---
old2 = '    ans_file = open(answers_file, "w")\n'
new2 = """    done_lines = []
    if os.path.exists(answers_file):
        for l in open(answers_file):
            try:
                json.loads(l)
                done_lines.append(l.strip())
            except Exception:
                pass
    done_ids = {json.loads(l)["id"] for l in done_lines}
    with open(answers_file, "w") as f:
        f.write("".join(l + "\\n" for l in done_lines))
    questions = [q for q in questions if q["id"] not in done_ids]
    print(f"[patch] resume: {len(done_ids)} done, {len(questions)} left", flush=True)
    ans_file = open(answers_file, "a")
"""
old3 = '        ans_file.flush()\n'
new3 = '        ans_file.close()\n        ans_file = open(answers_file, "a")\n'

if "M3DIFF_QUANT" not in src:
    assert old1 in src, "Không tìm thấy đoạn patch 1"
    src = src.replace(old1, new1)
if "[patch] resume" not in src:
    assert old2 in src and old3 in src, "Không tìm thấy đoạn patch 2"
    src = src.replace(old2, new2).replace(old3, new3)
open(path, "w").write(src)
print("Patch OK:", "M3DIFF_QUANT" in src, "[patch] resume" in src)

In [ ]:
from IPython.display import Image, display
import pandas as pd
os.makedirs(f"{REPO}/datasets", exist_ok=True)
link = f"{REPO}/datasets/ft_synth"          # ảnh train tổng hợp nằm trên Volume, nối vào thư mục datasets
if os.path.isdir(TRAIN_SYN) and not os.path.islink(link):
    os.symlink(TRAIN_SYN, link)

def run_eval(qfile, answers, image_folder, lora):
    mp = f"--model-path {OUT} --model-base {CKPT}" if lora else f"--model-path {CKPT}"
    cmd = (f"cd {REPO} && PYTHONWARNINGS=ignore M3DIFF_QUANT={QUANT} {PY} llava/eval/model_vqa_qwen_OmniDiff_Real.py "
           f"{mp} --question-file {qfile} --answers-file {answers} --image-folder {image_folder} "
           f"--num-chunks 1 --chunk-idx 0 --temperature 0 --torch-dtype {DTYPE} --no-flash-attention")
    n_total = len(json.load(open(qfile)))
    done = sum(1 for _ in open(answers)) if os.path.exists(answers) else 0
    if done >= n_total:
        print(f"Đã đủ {done}/{n_total}: {answers}"); return
    t0 = time.time()
    rc = subprocess.run(cmd, shell=True).returncode
    done = sum(1 for _ in open(answers)) if os.path.exists(answers) else 0
    print(f"-> {done}/{n_total} câu, {(time.time()-t0)/60:.1f} phút, mã thoát {rc}")

MDP_ARGS = '{"num_refinement_layers":2,"feature_dim":1152,"ffn_hidden_dim":4608,"attention_downsample_rate":1}'

def train_args(out_dir, extra=()):
    return [PY, f"{CODE}/ft_train.py",
        "--model_name_or_path", CKPT, "--version", "qwen_2", "--output_dir", out_dir,
        "--data_path", MIX, "--image_folder", f"{REPO}/datasets", "--image_aspect_ratio", "pad",
        "--vision_tower", "google/siglip-so400m-patch14-384", "--mm_vision_select_layer", "[-11,-8,-5,-2]",
        "--unfreeze_mm_vision_tower", "False", "--mm_projector_type", "mlp2x_gelu", "--mm_projector_lr", str(PROJ_LR),
        "--mm_patch_merge_type", "flat", "--mm_resampler_type", "mdp", "--mdp_args", MDP_ARGS,
        "--tune_mm_vision_resampler", "True", "--mm_use_im_start_end", "False", "--mm_use_im_patch_token", "False",
        "--lora_enable", "True", "--lora_r", str(LORA_R), "--lora_alpha", str(LORA_ALPHA),
        "--num_train_epochs", str(EPOCHS), "--per_device_train_batch_size", str(BATCH),
        "--gradient_accumulation_steps", str(ACCUM), "--learning_rate", str(LR), "--weight_decay", "0.",
        "--warmup_ratio", "0.03", "--lr_scheduler_type", "cosine", "--logging_steps", "5",
        "--model_max_length", "4096", "--gradient_checkpointing", "True", "--group_by_modality_length", "True",
        "--dataloader_num_workers", "4", "--lazy_preprocess", "True", "--bf16", "True", "--tf32", "True",
        "--report_to", "none", *extra]

def run(args):
    from collections import deque
    env = dict(os.environ, ATTN="sdpa", PYTHONWARNINGS="ignore", TOKENIZERS_PARALLELISM="false")
    t0, tail = time.time(), deque(maxlen=40)
    p = subprocess.Popen(args, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    keep = ("'loss'", "train_runtime", "out of memory", "resume", "it/s", "s/it", "Adding LoRA")
    for line in p.stdout:
        tail.append(line.rstrip())
        if any(k in line for k in keep):
            print(line.rstrip()[-300:], flush=True)
    p.wait()
    if p.returncode != 0:
        print("\n----- LỖI: 40 dòng cuối -----\n" + "\n".join(tail))
    print(f"[xong] mã thoát {p.returncode}, {(time.time()-t0)/60:.1f} phút")
    return p.returncode
print("OK")

In [ ]:
# Mandatory stop instead of allowing Step 11 to start training.
assert os.path.isfile(f'{OUT}/adapter_model.bin') or os.path.isfile(f'{OUT}/adapter_model.safetensors'), 'STOP: chưa có checkpoint đã train'
state=json.load(open(f'{OUT}/trainer_state.json'))
assert state['global_step']==state['max_steps']==338, 'STOP: training not complete'
if not os.path.islink(f"{REPO}/datasets/ft_synth"):
    os.symlink(TRAIN_SYN, f"{REPO}/datasets/ft_synth")
if os.path.exists(f"{OUT}/adapter_model.bin") or os.path.exists(f"{OUT}/adapter_model.safetensors"):
    print("Đã train xong:", OUT)
else:
    rc = run(train_args(OUT, ["--save_strategy", "steps", "--save_steps", "50", "--save_total_limit", "1"]))

# Đảm bảo khi nạp lại, vision tower lấy đúng trọng số đã fine-tune của M3Diff gốc (nạp sớm từ checkpoint)
cfg_p = f"{OUT}/config.json"
if os.path.exists(cfg_p):
    cfg = json.load(open(cfg_p)); cfg["unfreeze_mm_vision_tower"] = True
    json.dump(cfg, open(cfg_p, "w"), indent=2)
    print("Đã lưu model:", sorted(os.listdir(OUT)))

In [ ]:
RUN='/mnt/m3diff/r64_val_precheck_20261102'
import hashlib
lock=json.load(open(f'{RUN}/SELECTION_LOCK.json'))
assert hashlib.sha256(open(f'{RUN}/questions.json','rb').read()).hexdigest()==lock['hashes']['questions.json']
assert len(json.load(open(f'{RUN}/questions.json')))==200
path=f'{REPO}/llava/eval/model_vqa_qwen_OmniDiff_Real.py'
src=open(path).read()
assert '    # Load and shard the evaluation data.' in src
audit_code="""    model.eval()
    from collections import Counter
    gen_audit={'loaded_generation_config':model.generation_config.to_dict(),
       'effective_explicit_overrides':{'do_sample':False,'temperature':0,'top_p':None,'num_beams':1,'max_new_tokens':1024,'repetition_penalty':1.05},
       'model_name':model_name,'dtype_arg':args.torch_dtype,
       'mdp_projector_dtype_counts':dict(Counter(str(p.dtype) for n,p in model.named_parameters() if 'vision_resampler' in n or 'mm_projector' in n))}
    json.dump(gen_audit,open(args.answers_file+'.generation_audit.json','w'),indent=2)
    print('GENERATION_AUDIT',gen_audit,flush=True)
"""
src=src.replace('    # Load and shard the evaluation data.',audit_code+'    # Load and shard the evaluation data.')
assert 'max_new_tokens=args.max_new_tokens,' in src
src=src.replace('max_new_tokens=args.max_new_tokens,','max_new_tokens=args.max_new_tokens,\n                repetition_penalty=1.05,')
open(path,'w').write(src)
json.dump({'runner_sha256':hashlib.sha256(src.encode()).hexdigest(),'repo_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd='/mnt/m3diff/baseline_reproduction_v1/repo',text=True).strip(),'gpu':gpu},open(f'{RUN}/runtime.json','w'),indent=2)
timings={'gpu':gpu,'evals':[]}
for arm,lora in [('base',False),('r64',True)]:
    answers=f'{RUN}/{arm}.jsonl'
    assert not os.path.exists(answers), 'Do not mix preexisting captions into matched session'
    t0=time.time()
    run_eval(f'{RUN}/questions.json',answers,f'{RUN}/images',lora=lora)
    qs=json.load(open(f'{RUN}/questions.json'));rs=[json.loads(l) for l in open(answers) if l.strip()]
    assert [r['id'] for r in rs]==[q['id'] for q in qs] and all(r['caption'].strip() for r in rs)
    timings['evals'].append({'arm':arm,'n':len(rs),'wall_seconds':time.time()-t0})
    json.dump(timings,open(f'{RUN}/timings.json','w'),indent=2)


In [ ]:
subprocess.run([PY,"-c","from pathlib import Path\nimport json\nfrom pycocotools.coco import COCO\nfrom pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer\nfrom pycocoevalcap.cider.cider import Cider\np=Path('/mnt/m3diff/r64_val_precheck_20261102');result={};per=[]\nfor domain in ['real','render']:\n    coco=COCO(str(p/f'{domain}_reference.json'));ids=coco.getImgIds();result[domain]={}\n    for arm in ['base','r64']:\n        rows=[json.loads(l) for l in (p/f'{arm}.jsonl').read_text().splitlines() if l.strip()]\n        rows=[{'image_id':r['id'],'caption':r['caption']} for r in rows if r['id'] in ids]\n        assert len(rows)==100\n        res=coco.loadRes(rows);tok=PTBTokenizer()\n        gts=tok.tokenize({i:coco.imgToAnns[i] for i in ids});pred=tok.tokenize({i:res.imgToAnns[i] for i in ids})\n        score,scores=Cider().compute_score(gts,pred);result[domain][arm]={'n':100,'CIDEr_x100':float(score)*100}\n        per += [{'domain':domain,'arm':arm,'id':i,'CIDEr_x100':float(s)*100} for i,s in zip(ids,scores)]\n    result[domain]['delta_r64_minus_base']=result[domain]['r64']['CIDEr_x100']-result[domain]['base']['CIDEr_x100']\n(p/'metrics.json').write_text(json.dumps(result,indent=2));(p/'per_pair_cider.json').write_text(json.dumps(per,indent=2))\nprint(json.dumps(result,indent=2))\n"],check=True)
json.dump({'complete':True,'pairs':200,'arms':['base','r64'],'same_container':True,'v2_trained':False,'full_test_run':False},open(f'{RUN}/SUCCESS.json','w'),indent=2)
print('DONE matched val200, stop GPU.')
